# НИС «Основы анализа данных в Python»

*Алла Тамбовцева*

## Практикум 16. Логистическая регрессия: часть 1

### Описание данных и постановка задачи

В файле `flowers_two.csv` хранятся характеристики 186 фотографий, на которых изображены цветы:

* `R`: средняя интенсивность красного цвета (усредненное значение по всем пикселям);
* `G`: средняя интенсивность зеленого цвета;
* `B`: средняя интенсивность синего цвета;
* `Name`: название цветка (`foxglove` – наперстянка, `monkshood` – аконит).

Источник изображений – [Kaggle](https://www.kaggle.com/datasets/yousefmohamed20/oxford-102-flower-dataset).

**Задача** 

Научиться классифицировать изображения по их цветовым характеристикам (здесь упрощенный вариант, поскольку файлы с изображениями уже обработаны и мы имеем дело с обычным датафреймом привычной размерности).


**Мотивация выбора таких двух классов (официальная версия)** 

С одной стороны, растения схожи по форме, с другой стороны, они чаще всего разного цвета, поэтому информации о средней интенсивности базовых цветов должно хватить для вполне качественной классификации. 

**Мотивация выбора таких двух классов (настоящая)** 

Названия этих растений – см. ниже :)

<img src="https://github.com/allatambov/PyDat25/blob/main/fun-fan.png?raw=true"></img>

### Задания

Импортируем все необходимые библиотеки и функции:

In [1]:
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score 

Загрузим данные:

In [2]:
df = pd.read_csv("flowers_two.csv")

### Задача 1

Закодируйте признак `Name` при помощи `LabelEncoder`, результат сохраните в переменную `Class`. Рассчитайте среднее значение полученного признака.

In [4]:
le = LabelEncoder()
df["Class"] = le.fit_transform(df["Name"])
print(df["Class"].mean()) #  доля 1

0.24731182795698925


### Задача 2

Необходимо построить модель логистической регрессии:

$$
\hat{P}(y = 1) = \sigma(\omega_0 + \omega_1 \times R + \omega_2 \times B), 
$$

где $y$ – признак, получившийся при кодировании в предыдущей задаче.

a. Разделите выборку на тренировочную и тестовую выборку в соотношении 80 к 20, воспроизводимость (`random_state`) равна 24. Укажите стандартное отклонение `R` по тренировочной выборке.

In [5]:
X = df[["R", "B"]]
y = df["Class"]

print(X.head())
print(y.head())

            R           B
0  104.548590   91.985842
1  139.838266  137.757932
2  143.523616  116.187864
3  152.282690   86.979517
4  131.409302  121.153530
0    1
1    0
2    1
3    0
4    0
Name: Class, dtype: int64


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    random_state = 42)

In [7]:
print(X_train["R"].std())

23.84299249234773


b. Обучите модель на тренировочной выборке. Укажите, чему, согласно модели, равна вероятность того, что на фото изображен аконит ($y = 1)$, если `R` равен среднему значению по нашей выборке, а `B` – максимальному значению по нашей выборке.

In [8]:
model = LogisticRegression()
model.fit(X_train, y_train)

# наша выборка – исходные данные до разделения, 
# X и y

r = X["R"].mean()
b = X["B"].max()

# .predict_proba() возвращает предсказанные вероятности
# для 0 и 1 сразу

print(model.predict_proba([[r, b]]))

[[0.01448004 0.98551996]]


/opt/anaconda3/lib/python3.7/site-packages/sklearn/base.py:451: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  "X does not have valid feature names, but"


In [10]:
# если хотим вероятность только для 1, 
# заберем только второе значение из массива массивов [[]]

model.predict_proba([[r, b]])[:, 1]

/opt/anaconda3/lib/python3.7/site-packages/sklearn/base.py:451: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  "X does not have valid feature names, but"


array([0.98551996])

c. На тестовой выборке постройте матрицу ошибок (*confusion matrix*). На тестовой выборке рассчитайте значение точности (*accuracy*).

In [11]:
# предсказываем значения y для всех наблюдений в выборке,
# чтобы было с чем сравнивать y^

y_pred = model.predict(X_test)

# вычисляем метрики

print(confusion_matrix(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

[[30  1]
 [ 3  4]]
Accuracy: 0.8947368421052632


> Python автоматически принял пороговое значение вероятности равным 0.5, и в соответствии с этим в результат `.predict()` записал значения классов 0 и 1 ($\hat{y}$). Далее была построена матрица ошибок значения $y$ против $\hat{y}$. По ней видно, что $\text{TP}=30$, а $\text{TN}=4$, значит, точность равна 34/38, это и есть 0.8947.